# Charger booking: trace formulas

This version follows the canonical flow `LOGIC.md -> CODE.md -> code`. A booking request is a formula over physical time, and complete schedules use `Trace`.

In [ ]:
from dataclasses import dataclass

import portion as P
import interval_logic as il

## Build the request

The outer absolute eventuality chooses a start in the allowed interval. Its operand is evaluated locally at that witness, so the inner relative always condition starts there and measures an actual duration.

In [ ]:
@dataclass(frozen=True)
class Request:
    job: int
    duration: float
    start_interval: P.Interval

    @property
    def proposition(self) -> il.Formula:
        return il.Signal("job").equals(self.job)

    @property
    def formula(self) -> il.Formula:
        return self.proposition.always(self.duration).eventually(
            self.start_interval
        )

request = Request(2, 1.0, P.closedopen(3, 5))
print(request.formula)

## Check one complete schedule

The signal valuation is complete over the represented universe. The absolute start interval is future-truncated by the outer condition, while the relative dwell condition is re-anchored at each candidate start.

In [ ]:
trace = il.Trace(
    P.closed(0, 8),
    signals={
        "job": il.NaturalSignal({
            0: P.closedopen(0, 4) | P.openclosed(6, 8),
            2: P.closed(4, 6),
        })
    },
)

print("request holds at 0:", il.satisfies(request.formula, trace, 0))
print("satisfying anchors:", il.satisfying_times(request.formula, trace))